In [ ]:
!pip install langchain
!pip install langchain-community
!pip install langchain-text-splitters
!pip install sentence-transformers
!pip install transformers
!pip install faiss-cpu

In [ ]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("ml_notes.txt")

documents = loader.load()

print(documents)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=80
)

docs = text_splitter.split_documents(documents)

print("Number of chunks:", len(docs))

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings()

In [ ]:
from langchain_community.vectorstores import FAISS

db = FAISS.from_documents(docs, embeddings)

print("Vector database built")

In [ ]:
from transformers import pipeline

qa_model = pipeline("question-answering")

In [ ]:
import re

query = input("Ask a question: ")

results = db.similarity_search(query, k=5)

context = " ".join([doc.page_content for doc in results])

response = qa_model(
    question=query,
    context=context
)

answer = response["answer"]

# find the full sentence containing the answer
sentences = re.split(r'(?<=[.!?]) +', context)

full_sentence = next((s for s in sentences if answer in s), answer)

print("\nAnswer:")
print(full_sentence)